# OpenPlaque — Combined TPV + Canonical RCA PCAT (v2)

One self-contained Colab workflow that runs the established LAD/RCA/LCX plaque-volume analysis and the locked RCA 10–50 mm PCAT analysis, then creates consolidated summary tables.

**Change from v1:** the erosion-based `core TPV` interval has been removed because one-voxel 3-D erosion collapsed all plaque components to zero in this case. It is replaced by a **refinement-sensitivity range** across a prespecified 3×3 grid of post-processing settings. This is a non-statistical method-sensitivity analysis, not a confidence interval.

Research use only. OpenPlaque PCAT Attenuation is not Caristo FAI-Score.


In [ ]:
# FIRST EXECUTABLE CELL — mount Google Drive first.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch combined-tpv-pcat-sensitivity-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
print('Repository and requirements ready.')


In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy.spatial import cKDTree
from IPython.display import display, HTML
REPO=Path('/content/OpenPlaque'); sys.path.insert(0,str(REPO/'src'))
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'Combined_TPV_PCAT_All_Metrics_v2'; OUT.mkdir(parents=True,exist_ok=True)
os.environ['nnUNet_raw']='/content/nnUNet_raw'; os.environ['nnUNet_preprocessed']='/content/nnUNet_preprocessed'; os.environ['nnUNet_results']='/content/nnUNet_results'
for d in [os.environ['nnUNet_raw'],os.environ['nnUNet_preprocessed'],os.environ['nnUNet_results']]: Path(d).mkdir(parents=True,exist_ok=True)
model_zip=ROOT/'models'/'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
model_target=Path('/content/nnUNet_results/Dataset001_CCTA_DHM')
if not model_target.exists():
    if not model_zip.exists(): raise FileNotFoundError(model_zip)
    with zipfile.ZipFile(model_zip) as z: z.extractall('/content/nnUNet_results')
drive_zip=ROOT/'Full_DICOM.zip'; local_zip=Path('/content/Full_DICOM.zip')
if not local_zip.exists() or local_zip.stat().st_size!=drive_zip.stat().st_size: shutil.copyfile(drive_zip,local_zip)
from openplaque.study import OpenPlaqueStudy
shutil.rmtree('/content/full_dicom_combined_v2',ignore_errors=True)
study=OpenPlaqueStudy(str(local_zip),extract_root='/content/full_dicom_combined_v2')
print('Output:',OUT)


## Part A — Total plaque volume

Canonical refinement uses `min_component_voxels=10` and `lumen_distance_voxels=1`. Sensitivity is evaluated over `min_component_voxels ∈ {5,10,20}` × `lumen_distance_voxels ∈ {0,1,2}`. The same parameter pair is applied to all three vessels for each total-TPV sensitivity run.


In [ ]:
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask
from openplaque.artery_detection import detect_artery_series
fallback={'RCA':1035,'LCX':1039,'LAD':1043}
series_map,_=detect_artery_series(study,fallback_series=fallback,return_candidates=True)
print('Detected series:',series_map)
reports=[]
for vessel in ['LAD','RCA','LCX']:
    image,volume,_=study.load_series(series_map[vessel])
    print('Segmenting',vessel,'series',series_map[vessel],volume.shape)
    r=segment_vessel(image,volume,vessel); reports.append(r); r.summary()
report_map={r.name:r for r in reports}
def refine(report,min_component_voxels=10,lumen_distance_voxels=1):
    return refine_plaque_mask(volume=report.volume,mask=report.mask,spacing=report.mask_image.GetSpacing(),remove_small=True,min_component_voxels=int(min_component_voxels),trim_lumen_adjacent=True,lumen_distance_voxels=int(lumen_distance_voxels),erode_core=False,high_hu_threshold=None,low_hu_threshold=None)
canonical={r.name:refine(r,10,1) for r in reports}
grid=[]
for mc in [5,10,20]:
    for ld in [0,1,2]:
        row={'min_component_voxels':mc,'lumen_distance_voxels':ld}
        total=0.0
        for r in reports:
            x=refine(r,mc,ld); row[f'{r.name}_tpv_mm3']=x.refined_tpv_mm3; total+=x.refined_tpv_mm3
        row['TOTAL_tpv_mm3']=total; grid.append(row)
tpv_sens=pd.DataFrame(grid); tpv_sens.to_csv(OUT/'tpv_refinement_sensitivity_grid.csv',index=False)
rows=[]
for vessel in ['LAD','RCA','LCX']:
    r=report_map[vessel]; c=canonical[vessel]; vals=tpv_sens[f'{vessel}_tpv_mm3'].to_numpy(float)
    rows.append({'vessel':vessel,'raw_tpv_mm3':r.tpv_mm3,'canonical_refined_tpv_mm3':c.refined_tpv_mm3,'removed_mm3':r.tpv_mm3-c.refined_tpv_mm3,'removed_pct':100*(r.tpv_mm3-c.refined_tpv_mm3)/r.tpv_mm3 if r.tpv_mm3 else np.nan,'sensitivity_min_mm3':vals.min(),'sensitivity_max_mm3':vals.max(),'sensitivity_width_mm3':vals.max()-vals.min(),'sensitivity_max_abs_delta_mm3':np.max(np.abs(vals-c.refined_tpv_mm3)),'raw_plaque_voxels':r.plaque_voxels,'refined_plaque_voxels':c.refined_plaque_voxels})
raw_total=sum(r.tpv_mm3 for r in reports); refined_total=sum(canonical[r.name].refined_tpv_mm3 for r in reports); tv=tpv_sens.TOTAL_tpv_mm3.to_numpy(float)
rows.append({'vessel':'TOTAL','raw_tpv_mm3':raw_total,'canonical_refined_tpv_mm3':refined_total,'removed_mm3':raw_total-refined_total,'removed_pct':100*(raw_total-refined_total)/raw_total,'sensitivity_min_mm3':tv.min(),'sensitivity_max_mm3':tv.max(),'sensitivity_width_mm3':tv.max()-tv.min(),'sensitivity_max_abs_delta_mm3':np.max(np.abs(tv-refined_total)),'raw_plaque_voxels':sum(r.plaque_voxels for r in reports),'refined_plaque_voxels':sum(canonical[r.name].refined_plaque_voxels for r in reports)})
tpv=pd.DataFrame(rows); tpv.to_csv(OUT/'tpv_metrics_by_vessel_v2.csv',index=False)
display(tpv); display(tpv_sens)


In [ ]:
# Compact TPV QC figure
fig,axs=plt.subplots(1,3,figsize=(15,5))
for ax,r in zip(axs,reports):
    m=canonical[r.name].refined_mask==2; counts=np.sum(m,axis=(1,2)); z=int(np.argmax(counts)) if np.any(counts) else r.volume.shape[0]//2
    ax.imshow(r.volume[z],cmap='gray',vmin=-200,vmax=800); ax.contour(m[z],levels=[0.5],linewidths=1); ax.set_title(f'{r.name}: {canonical[r.name].refined_tpv_mm3:.0f} mm³'); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'01_tpv_refined_qc_v2.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,5))
for vessel in ['LAD','RCA','LCX','TOTAL']:
    ax.plot(range(len(tpv_sens)),tpv_sens[f'{vessel}_tpv_mm3'],marker='o',label=vessel)
ax.set_xticks(range(len(tpv_sens))); ax.set_xticklabels([f"{a}/{b}" for a,b in zip(tpv_sens.min_component_voxels,tpv_sens.lumen_distance_voxels)],rotation=45,ha='right'); ax.set_ylabel('TPV mm³'); ax.set_xlabel('min component voxels / lumen distance voxels'); ax.set_title('TPV refinement sensitivity'); ax.legend(); plt.tight_layout(); plt.savefig(OUT/'02_tpv_refinement_sensitivity.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


## Part B — Locked RCA 10–50 mm PCAT


In [ ]:
BASE=ROOT/'PCAT_RCA_10_50'; cp=BASE/'rca_centerline_smoothed_zyx.csv'; rp=BASE/'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists(): raise FileNotFoundError('Missing frozen PCAT centerline/radius inputs.')
source_img,ct,_=study.load_series(7); ct=np.asarray(ct); sp_xyz=np.array(source_img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]; voxel_mm3=float(np.prod(sp_xyz))
cl=pd.read_csv(cp); rad=pd.read_csv(rp); arc=cl.arc_mm.to_numpy(float); pts=cl[['z','y','x']].to_numpy(float); pts_mm=pts*sp_zyx; lumen_all=np.interp(arc,rad.arc_mm.to_numpy(float),rad.lumen_radius_mm.to_numpy(float))
SEG0,SEG1=10.,50.; FAT_LO,FAT_HI=-190.,-30.; MARGINS=[0.25,0.50,0.75,1.00,1.25]; PRIMARY=0.75
sm=(arc>=SEG0)&(arc<=SEG1); seg_arc=arc[sm]; seg_zyx=pts[sm]; seg_mm=pts_mm[sm]; seg_lumen=lumen_all[sm]
max_outer=float(np.max(seg_lumen+max(MARGINS))); pad=3*max_outer+3
lo=np.maximum(np.floor(np.min(seg_zyx,axis=0)-pad/sp_zyx).astype(int),0); hi=np.minimum(np.ceil(np.max(seg_zyx,axis=0)+pad/sp_zyx).astype(int)+1,np.array(ct.shape))
crop=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]; zz,yy,xx=np.indices(crop.shape); g=np.stack([zz+lo[0],yy+lo[1],xx+lo[2]],axis=-1).reshape(-1,3).astype(float); gmm=g*sp_zyx
tree=cKDTree(seg_mm); dist,ni=tree.query(gmm,k=1,workers=-1); ni=ni.astype(int); nearest_arc=seg_arc[ni]; nearest_lumen=seg_lumen[ni]; hu=crop.reshape(-1).astype(float); fat_hu=(hu>=FAT_LO)&(hu<=FAT_HI)
aorta_candidates=[ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz']; ap=next((p for p in aorta_candidates if p.exists()),None)
if ap is None: raise FileNotFoundError('Missing cached TotalSegmentator aorta mask.')
ai=sitk.ReadImage(str(ap));
if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(),source_img.GetSpacing()): ai=sitk.Resample(ai,source_img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(ai)>0; aorta_flat=aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].reshape(-1)
def pcat(margin):
    outer=nearest_lumen+float(margin); shell_outer=3*outer; shell=(dist>outer)&(dist<=shell_outer)&(~aorta_flat); fat=shell&fat_hu; vals=hu[fat]; radial_out=dist-outer
    radial=[]
    for b in np.arange(0,6,0.5):
        q=fat&(radial_out>=b)&(radial_out<b+0.5); vv=hu[q]; radial.append({'radial_start_mm':b,'radial_end_mm':b+0.5,'fat_voxels':int(q.sum()),'mean_hu':float(np.mean(vv)) if len(vv) else np.nan})
    longitudinal=[]
    for b in range(10,50):
        q=fat&(nearest_arc>=b)&(nearest_arc<b+1); vv=hu[q]; longitudinal.append({'arc_start_mm':b,'arc_end_mm':b+1,'fat_voxels':int(q.sum()),'mean_hu':float(np.mean(vv)) if len(vv) else np.nan})
    return {'wall_margin_mm':float(margin),'pcat_mean_hu':float(np.mean(vals)),'pcat_median_hu':float(np.median(vals)),'pcat_sd_hu':float(np.std(vals)),'fat_voxels':int(fat.sum()),'fat_volume_ml':float(fat.sum()*voxel_mm3/1000),'shell_voxels':int(shell.sum()),'shell_volume_ml':float(shell.sum()*voxel_mm3/1000),'fat_fraction':float(fat.sum()/max(1,shell.sum())),'mean_lumen_radius_mm':float(np.mean(seg_lumen)),'radial':pd.DataFrame(radial),'longitudinal':pd.DataFrame(longitudinal)}
primary=pcat(PRIMARY); ps=[pcat(x) for x in MARGINS]; pcat_sens=pd.DataFrame([{k:v for k,v in r.items() if k not in ('radial','longitudinal')} for r in ps]); pcat_sens.to_csv(OUT/'pcat_circular_sensitivity_v2.csv',index=False)
primary_df=pd.DataFrame([{k:v for k,v in primary.items() if k not in ('radial','longitudinal')}]); primary_df.to_csv(OUT/'pcat_canonical_primary_v2.csv',index=False); primary['radial'].to_csv(OUT/'pcat_canonical_radial_v2.csv',index=False); primary['longitudinal'].to_csv(OUT/'pcat_canonical_longitudinal_v2.csv',index=False)
display(primary_df.T); display(pcat_sens)


In [ ]:
directional=np.nan; dp=ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_summary.csv'
if dp.exists():
    d=pd.read_csv(dp); directional=float(d.iloc[0]['directional_pcat_mean_hu']) if 'directional_pcat_mean_hu' in d.columns else np.nan
methods=pcat_sens[['wall_margin_mm','pcat_mean_hu','fat_voxels','fat_volume_ml']].copy(); methods['method']=methods.wall_margin_mm.map(lambda x:f'circular +{x:.2f} mm')
if np.isfinite(directional): methods=pd.concat([methods,pd.DataFrame([{'wall_margin_mm':np.nan,'pcat_mean_hu':directional,'fat_voxels':np.nan,'fat_volume_ml':np.nan,'method':'directional fat-interface'}])],ignore_index=True)
methods['delta_from_canonical_hu']=methods.pcat_mean_hu-primary['pcat_mean_hu']; methods.to_csv(OUT/'pcat_geometry_method_comparison_v2.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(pcat_sens.wall_margin_mm,pcat_sens.pcat_mean_hu,marker='o',label='circular sensitivity'); ax.axhline(primary['pcat_mean_hu'],linestyle='--',label='canonical +0.75 mm');
if np.isfinite(directional): ax.axhline(directional,linestyle=':',label='directional interface')
ax.set_xlabel('outer-wall margin mm'); ax.set_ylabel('PCAT mean HU'); ax.set_title('PCAT geometry sensitivity'); ax.legend(); plt.tight_layout(); plt.savefig(OUT/'03_pcat_geometry_sensitivity_v2.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
display(methods)


## Final consolidated summary tables


In [ ]:
T=tpv.set_index('vessel'); method_vals=methods.pcat_mean_hu.to_numpy(float)
summary=pd.DataFrame([{'total_raw_tpv_mm3':T.loc['TOTAL','raw_tpv_mm3'],'total_refined_tpv_mm3':T.loc['TOTAL','canonical_refined_tpv_mm3'],'total_removed_mm3':T.loc['TOTAL','removed_mm3'],'total_removed_pct':T.loc['TOTAL','removed_pct'],'tpv_refinement_sensitivity_min_mm3':T.loc['TOTAL','sensitivity_min_mm3'],'tpv_refinement_sensitivity_max_mm3':T.loc['TOTAL','sensitivity_max_mm3'],'tpv_refinement_sensitivity_width_mm3':T.loc['TOTAL','sensitivity_width_mm3'],'tpv_refinement_max_abs_delta_mm3':T.loc['TOTAL','sensitivity_max_abs_delta_mm3'],'lad_refined_tpv_mm3':T.loc['LAD','canonical_refined_tpv_mm3'],'rca_refined_tpv_mm3':T.loc['RCA','canonical_refined_tpv_mm3'],'lcx_refined_tpv_mm3':T.loc['LCX','canonical_refined_tpv_mm3'],'rca_pcat_mean_hu_10_50mm':primary['pcat_mean_hu'],'rca_pcat_median_hu':primary['pcat_median_hu'],'rca_pcat_sd_hu':primary['pcat_sd_hu'],'rca_pcat_fat_voxels':primary['fat_voxels'],'rca_pcat_fat_volume_ml':primary['fat_volume_ml'],'rca_pcat_shell_volume_ml':primary['shell_volume_ml'],'rca_pcat_fat_fraction':primary['fat_fraction'],'rca_mean_lumen_radius_mm':primary['mean_lumen_radius_mm'],'pcat_geometry_min_hu':float(np.nanmin(method_vals)),'pcat_geometry_max_hu':float(np.nanmax(method_vals)),'pcat_geometry_range_hu':float(np.nanmax(method_vals)-np.nanmin(method_vals)),'pcat_geometry_max_abs_delta_hu':float(np.nanmax(np.abs(method_vals-primary['pcat_mean_hu'])))}])
summary.to_csv(OUT/'subject_summary_all_metrics_v2.csv',index=False)
long_rows=[]
for _,r in tpv.iterrows():
    for metric in ['raw_tpv_mm3','canonical_refined_tpv_mm3','removed_mm3','removed_pct','sensitivity_min_mm3','sensitivity_max_mm3','sensitivity_width_mm3','sensitivity_max_abs_delta_mm3']:
        long_rows.append({'domain':'TPV','scope':r.vessel,'metric':metric,'value':r[metric]})
for k,v in summary.iloc[0].items(): long_rows.append({'domain':'SUMMARY','scope':'subject','metric':k,'value':v})
all_long=pd.DataFrame(long_rows); all_long.to_csv(OUT/'all_metrics_long_v2.csv',index=False)
print('=== TPV BY VESSEL ==='); display(tpv)
print('=== TPV REFINEMENT SENSITIVITY GRID ==='); display(tpv_sens)
print('=== CANONICAL PCAT ==='); display(primary_df)
print('=== PCAT GEOMETRY SENSITIVITY ==='); display(methods)
print('=== SINGLE-ROW ALL-METRICS SUMMARY ==='); display(summary.T)


In [ ]:
# Package report-back files and print direct Drive search links.
import zipfile
zip_path=OUT/'OPENPLAQUE_COMBINED_TPV_PCAT_V2_REPORT_BACK.zip'
files=[OUT/'tpv_metrics_by_vessel_v2.csv',OUT/'tpv_refinement_sensitivity_grid.csv',OUT/'pcat_canonical_primary_v2.csv',OUT/'pcat_circular_sensitivity_v2.csv',OUT/'pcat_geometry_method_comparison_v2.csv',OUT/'pcat_canonical_radial_v2.csv',OUT/'pcat_canonical_longitudinal_v2.csv',OUT/'subject_summary_all_metrics_v2.csv',OUT/'all_metrics_long_v2.csv',OUT/'01_tpv_refined_qc_v2.png',OUT/'02_tpv_refinement_sensitivity.png',OUT/'03_pcat_geometry_sensitivity_v2.png']
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files:
        if p.exists(): z.write(p,arcname=p.name)
print('Report ZIP:',zip_path)
print('https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_COMBINED_TPV_PCAT_V2_REPORT_BACK.zip')
print('https://drive.google.com/drive/u/0/search?q=subject_summary_all_metrics_v2.csv')
